# Mini-TP 2 — Los metadatos del modelo por GraphQL

Operaciones de Aprendizaje Automático II · CEIA – FIUBA · Entrega individual

Expone por GraphQL los metadatos del mismo modelo que la Sesión 1 sirve por REST: XGBoost que
predice si un crimen reportado en Chicago (2024) termina en arresto.

## Qué se entrega

1. Un esquema GraphQL con Strawberry, con un tipo `Model` que tiene nombre, versión y métricas.
2. Una query que devuelve esas métricas.
3. La prueba desde GraphiQL y desde un cliente Python.
4. La comparación de la misma lectura contra el endpoint REST, y la diferencia observada.

Como extra, un resolver que lee el linaje del modelo desde Neo4j.

## Cómo está organizado

El código vive en los `.py` del repo (`tp2_graphql/schema.py`, `app.py`, `client.py`,
`compare.py`, `lineage.py`) y está cubierto por tests; este notebook los importa, los muestra y
los ejecuta. De esa forma no hay dos copias del mismo código que se desincronicen.

Las métricas salen de `model/model.pkl`, no de MLflow: son las mismas que ya sirve la API REST,
y la consigna acepta que vengan de un registro local o simuladas.

In [1]:
import logging
import os
from pathlib import Path

import arrest_model

# El notebook vive en tp2_graphql/, pero los imports esperan la raíz del repo.
REPO = Path(arrest_model.__file__).resolve().parent.parent
os.chdir(REPO)

# La API REST del TP1 deja una línea de log por request, y acá corre en este mismo proceso:
# esas líneas irían a la salida de las celdas. Se silencia desde el arranque.
logging.getLogger("tp1_rest").setLevel(logging.WARNING)

print("Trabajando desde:", REPO)

Trabajando desde: /Users/belcattaneo/Documents/MIA/MLOPS2/mlops2-minitps


## 1. Los metadatos del modelo

Salen del bundle que comparten las tres APIs del repo. GraphQL no reimplementa nada del modelo:
traduce este diccionario al esquema.

In [2]:
from arrest_model.model import load_bundle

bundle = load_bundle()
metadata = bundle["metadata"]
print(f"{metadata['name']} v{metadata['version']} ({metadata['framework']})")
print("métricas:", {k: round(v, 4) for k, v in metadata["metrics"].items()})

chicago-arrest-xgboost v1 (xgboost 3.4.1)
métricas: {'accuracy': 0.9119, 'precision': 0.9047, 'recall': 0.9119, 'f1': 0.9039, 'auc': 0.8861, 'mcc': 0.5811}


## 2. El esquema GraphQL

El esquema declara qué campos existen y de qué tipo son. El cliente arma su propio recorte: pide
los campos que necesita y recibe exactamente eso.

`lineage` es el único campo que sale de Neo4j, y solo se consulta si la query lo pide.

In [3]:
from IPython.display import Code, display

display(Code(filename="tp2_graphql/schema.py", language="python"))

"""Mini-TP 2: esquema GraphQL con los metadatos del modelo de arrestos de Chicago.

Los datos salen del mismo `model/model.pkl` que sirven REST y gRPC: acá no se reimplementa
nada del modelo, solo se traduce su metadata al esquema.

El bundle no se carga al importar el módulo, entra por el contexto de la query. Así el esquema
se puede ejecutar en los tests sin levantar un servidor, y la app lo inyecta al arrancar.
"""

from datetime import datetime
from typing import Any

import strawberry

from tp2_graphql.lineage import lineage_of


@strawberry.type
class Metrics:
    """Métricas del modelo sobre el conjunto de test del TP-final."""

    accuracy: float
    precision: float
    recall: float
    f1: float
    auc: float
    mcc: float


@strawberry.type
class Artifact:
    """Un artefacto aguas arriba del modelo: un dataset o una transformación."""

    name: str
    kind: str


@strawberry.type
class Model:
    """El modelo que están sirviendo las APIs del repo."""

    name: str
    version: int
    framework: str
    inputs: list[str]
    features: list[str]
    trained_at: datetime
    metrics: Metrics

    @strawberry.field
    def lineage(self, info: strawberry.Info) -> list[Artifact] | None:
        """De dónde salió el modelo: sus datasets y transformaciones, leídos de Neo4j.

        Solo se consulta si la query pide este campo; ese es justamente el punto de GraphQL.
        Es nullable a propósito: si Neo4j no responde, el error queda acá y el resto de la
        respuesta llega igual.
        """
        artefactos = lineage_of(info.context["driver"], self.name)
        return [Artifact(name=a["name"], kind=a["kind"]) for a in artefactos]


def model_from_metadata(metadata: dict[str, Any]) -> Model:
    """Arma el tipo del esquema a partir de la metadata que viaja en el bundle."""
    return Model(
        name=metadata["name"],
        version=metadata["version"],
        framework=metadata["framework"],
        inputs=list(metadata["inputs"]),
        features=list(metadata["features"]),
        trained_at=datetime.fromisoformat(metadata["trained_at"]),
        metrics=Metrics(**metadata["metrics"]),
    )


@strawberry.type
class Query:
    """Punto de entrada del esquema."""

    @strawberry.field
    def model(self, name: str, info: strawberry.Info) -> Model | None:
        """Devuelve el modelo con ese nombre, o null si no es el que está cargado."""
        metadata = info.context["bundle"]["metadata"]
        return model_from_metadata(metadata) if metadata["name"] == name else None


schema = strawberry.Schema(query=Query)

In [4]:
from tp2_graphql.schema import schema

print(schema.as_str())

type Artifact {
  name: String!
  kind: String!
}

"""Date with time (isoformat)"""
scalar DateTime

type Metrics {
  accuracy: Float!
  precision: Float!
  recall: Float!
  f1: Float!
  auc: Float!
  mcc: Float!
}

type Model {
  name: String!
  version: Int!
  framework: String!
  inputs: [String!]!
  features: [String!]!
  trainedAt: DateTime!
  metrics: Metrics!
  lineage: [Artifact!]
}

type Query {
  model(name: String!): Model
}


## 3. La API y un cliente Python

La app monta el esquema en `/graphql` con GraphiQL. Acá se levanta en un hilo de este mismo
proceso para poder consultarla desde el notebook; en la línea de comandos se levanta con
`make graphql-run` y se prueba desde el navegador.

In [5]:
import threading
import time

import requests
import uvicorn

from tp2_graphql.app import create_app

GRAPHQL_URL = "http://127.0.0.1:8010/graphql"
graphql_server = uvicorn.Server(
    uvicorn.Config(create_app(), host="127.0.0.1", port=8010, log_level="warning")
)
threading.Thread(target=graphql_server.run, daemon=True).start()

for _ in range(100):
    try:
        if requests.get(GRAPHQL_URL, headers={"Accept": "text/html"}, timeout=1).ok:
            break
    except requests.RequestException:
        time.sleep(0.1)
print("API GraphQL lista en el puerto 8010")

API GraphQL lista en el puerto 8010


In [6]:
QUERY = '{ model(name: "chicago-arrest-xgboost") { name version metrics { mcc auc } } }'

respuesta = requests.post(GRAPHQL_URL, json={"query": QUERY}, timeout=10)
print(respuesta.status_code, respuesta.json())

200 {'data': {'model': {'name': 'chicago-arrest-xgboost', 'version': 1, 'metrics': {'mcc': 0.5811392832702954, 'auc': 0.8861377061145108}}}}


In [7]:
# Se pide una sola métrica: la respuesta trae esa y nada más.
UNA_SOLA = '{ model(name: "chicago-arrest-xgboost") { metrics { mcc } } }'
print(requests.post(GRAPHQL_URL, json={"query": UNA_SOLA}, timeout=10).json())

# Un modelo que no existe devuelve null, sin error.
print(requests.post(GRAPHQL_URL, json={"query": '{ model(name: "otro") { name } }'},
                    timeout=10).json())

{'data': {'model': {'metrics': {'mcc': 0.5811392832702954}}}}
{'data': {'model': None}}


## 4. La comparación con REST

La vista a armar es chica: el nombre del modelo y una sola métrica, el MCC.

Por REST hay que pedir `/v1/metadata`, que devuelve el paquete entero —versión, framework, los 6
campos de entrada, las 7 features, las 6 métricas y la fecha— cuando solo se querían dos campos.
Por GraphQL se piden esos dos campos y no llega nada más.

Se mide en llamadas y en bytes recibidos.

In [8]:
from tp1_rest.app import app as rest_app
from tp2_graphql.compare import compare, format_results

REST_URL = "http://127.0.0.1:8000"
rest_server = uvicorn.Server(
    uvicorn.Config(rest_app, host="127.0.0.1", port=8000, log_level="warning")
)
threading.Thread(target=rest_server.run, daemon=True).start()

for _ in range(100):
    try:
        if requests.get(f"{REST_URL}/health", timeout=1).ok:
            break
    except requests.RequestException:
        time.sleep(0.1)
print("API REST lista en el puerto 8000")

API REST lista en el puerto 8000


In [9]:
print(format_results(compare(REST_URL, GRAPHQL_URL)))

Vista: nombre del modelo y su MCC.

| protocolo | llamadas | bytes recibidos |
|---|---|---|
| REST | 1 | 557 |
| GraphQL | 1 | 89 |

Las dos vistas coinciden: True.
El linaje del modelo no lo expone ningún endpoint REST; en GraphQL es un campo más.


## 5. El linaje, desde Neo4j

El linaje es la trazabilidad del modelo: de qué archivos crudos salió y por qué transformaciones
pasó. Se guarda como grafo porque la pregunta que interesa —qué hay aguas arriba de este modelo—
tiene largo variable, y eso en un grafo es un recorrido de una línea.

Necesita Neo4j corriendo (`make neo4j-up` y `make graphql-seed`). Si no está, la celda lo informa
y el resto del notebook sigue: el campo es nullable justamente para eso.

In [10]:
LINAJE = '''
{ model(name: "chicago-arrest-xgboost") { name lineage { name kind } } }
'''

cuerpo = requests.post(GRAPHQL_URL, json={"query": LINAJE}, timeout=60).json()
artefactos = cuerpo["data"]["model"]["lineage"]
if artefactos is None:
    print("Neo4j no respondió; el resto de la respuesta llegó igual:", cuerpo["data"]["model"])
else:
    print(f"{len(artefactos)} artefactos aguas arriba del modelo:")
    for artefacto in sorted(artefactos, key=lambda a: (a["kind"], a["name"])):
        print(f"  {artefacto['kind']:10} {artefacto['name']}")

16 artefactos aguas arriba del modelo:
  Dataset    Crimes_Chicago_2024.csv
  Dataset    Police_Stations_20251005.csv
  Dataset    chicago_crimes_and_stations_2024_final.csv
  Dataset    chicago_crimes_and_stations_2024_final_test.csv
  Dataset    chicago_crimes_and_stations_2024_processed.csv
  Dataset    chicago_crimes_and_stations_2024_processed_outliers_encoded.csv
  Dataset    chicago_crimes_and_stations_2024_processed_outliers_encoded_standardized.csv
  Dataset    chicago_crimes_and_stations_2024_processed_outliers_encoded_standardized_combined.csv
  Dataset    chicago_crimes_and_stations_2024_processed_outliers_encoded_standardized_test.csv
  Dataset    chicago_crimes_and_stations_2024_processed_outliers_encoded_test.csv
  Transform  1_Creacion_dataset
  Transform  3_Outliers_Encoding
  Transform  4_Escalado
  Transform  5_Balanceo
  Transform  6_Feature_Selection
  Transform  entrenamiento


## 6. Reflexión

> Pendiente de completar: la diferencia observada entre REST y GraphQL.

Lo medido, para apoyarse:

- La misma vista (nombre del modelo y su MCC) cuesta una llamada por cada protocolo, pero REST
  devuelve el payload entero y GraphQL solo los dos campos pedidos. Los bytes están en la tabla
  de la sección 4.
- El linaje no lo expone ningún endpoint REST: sumarlo a la vista obligaría a crear uno. En
  GraphQL es un campo más en la misma query, resuelto contra otra fuente de datos.
- El costo: hay un solo endpoint y un esquema que mantener, el cliente tiene que saber qué pedir,
  y una query mal armada falla en tiempo de ejecución con un 200 y una clave `errors`.

A responder: ¿qué diferencia noté entre REST y GraphQL para esta lectura?

## 7. Limpieza

In [11]:
graphql_server.should_exit = True
rest_server.should_exit = True
print("APIs detenidas")

APIs detenidas
